
# 練習問題: ファイル入出力

## 目標

ファイルを開いて読み込み, 終端まで繰り返し読むループの書き方を身につける.

## 課題

`fileio.cpp` (または `fileio.f90`) は, まず `data.txt` に 5 行の数値 (`i` と `i*0.5`) を書き出す (この部分は完成済み).
そのあと `data.txt` を読み直して, 2列目 `x` の合計を求めたい.
読み込みループが空なので, 現状の合計は `0`.

`TODO` の箇所に **1行ずつ読み, 読めた間 `x` を `sum` に足し込むループ** を書け.

- C++: `while (fscanf(rp, "%d %lf", &i, &x) == 2) { sum += x; }` (2個読めた間繰り返す)
- Fortran: `do` … `read(10, *, iostat=ios) i, x` … `if (ios /= 0) exit` … `sum = sum + x` … `end do` (`iostat` が 0 でなくなったら終端)

## コンパイルと実行

```
# C++
nvc++ -fast fileio.cpp -o fileio.exe

# Fortran
nvfortran -fast fileio.f90 -o fileio.exe
```

```
./fileio.exe
cat data.txt   # 書き出された中身を確認
```

## 期待される結果

`x` は `0.0, 0.5, 1.0, 1.5, 2.0` なので合計は 5.0.

```
sum of x = 5.000000
```

読み込みループを書く前は `0` になる. `data.txt` の中身も `cat` で確認せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ fileio.cpp
#include <cstdio>

int main() {
  /* まず data.txt に 0..4 の i と i*0.5 を書き出す (この部分は完成済み) */
  FILE * wp = fopen("data.txt", "w");
  if (wp == NULL) { printf("cannot open for write\n"); return 1; }
  for (int i = 0; i < 5; i++) {
    fprintf(wp, "%d %f\n", i, i * 0.5);
  }
  fclose(wp);

  /* 次に data.txt を読み直し, 2列目 (x) の合計を求める */
  FILE * rp = fopen("data.txt", "r");
  if (rp == NULL) { printf("cannot open for read\n"); return 1; }
  int i;
  double x;
  double sum = 0.0;
  // TODO: fscanf で1行ずつ (i, x) を読み, x を sum に足し込むループを書け.
  fclose(rp);
  printf("sum of x = %f\n", sum);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore fileio.cpp -o fileio_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./fileio_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ fileio.f90
program fileio
  integer :: i, ios
  real(8) :: x, sum
  ! まず data.txt に 0..4 の i と i*0.5 を書き出す (この部分は完成済み)
  open(unit=10, file="data.txt", status="replace", action="write")
  do i = 0, 4
     write(10, "(i0,1x,f0.6)") i, i * 0.5d0
  end do
  close(10)

  ! 次に data.txt を読み直し, 2列目 (x) の合計を求める
  open(unit=10, file="data.txt", status="old", action="read")
  sum = 0.0d0
  ! TODO: read で1行ずつ (i, x) を読み (iostat で終端判定), x を sum に足し込むループを書け.
  close(10)
  print "(a,f0.6)", "sum of x = ", sum
end program fileio

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore fileio.f90 -o fileio_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./fileio_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:fileio.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:fileio.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:fileio.cpp}